# SLR SLR: train the GrocerEye-style detector on Colab (TR-3)

Stage A: reproduce GrocerEye (YOLOv3, 84.59% mAP) with YOLO11 nano on the 5K Groceries / Freiburg
dataset, then evaluate on the held-out test split and download `best.pt` + the eval report.

1. **Runtime -> Change runtime type -> GPU** (T4 is fine).
2. Run the cells top to bottom. 100 epochs at imgsz 320 takes roughly 30-60 min on a T4.

### How to read this notebook (for non-programmers)

This notebook is a step-by-step recipe that runs the scripts in the `training/` folder on a
free Google Colab GPU. Each gray box is a *cell*: click it and press **Shift+Enter** to run it.
Inside code cells, lines beginning with `#` are comments (notes for humans, ignored by the
computer), lines beginning with `!` are terminal commands, and lines beginning with `%` are
notebook commands.

Words you will see:

- **Detector / model**: the neural network that finds grocery items in a camera frame.
- **YOLO11 nano**: the small, fast detector architecture we use (via the Ultralytics library).
- **Weights / `best.pt`**: the file holding the trained model's learned numbers.
- **Dataset / split**: the labeled photos, divided into *train* (to learn from), *val*
  (to monitor progress) and *test* (kept aside for an honest final score).
- **Epoch**: one full pass through all training photos.
- **mAP50**: the standard accuracy score for detectors (higher is better, max 1.0).
- **GPU / T4**: the graphics card that makes training fast; T4 is Colab's free one.
- **Stage A / Stage B**: training on the public dataset only / adding our own photos.
- **TR-3, NFR-5, ...**: requirement numbers from the product spec, `PRD.pdf`.

## 1. Get the code

The repo `Ore-Porat/VisuallyImpairedBodycam` is **private**, so a plain `git clone` fails. Either:

- **Token:** create a GitHub fine-grained personal access token with read access to the repo
  (GitHub -> Settings -> Developer settings), then run the next cell and paste it when asked
  (it is not saved in the notebook), or
- **Zip upload:** on your laptop zip the repo (without `.venv/`, `data/`, `runs/`), upload it with
  the Files panel on the left, set `USE_ZIP = True` below and put the file name in `ZIP_NAME`.

In [ ]:
# --- Get a copy of the project onto this Colab machine ---------------------------
# Colab is a free Google service that runs Python notebooks on a borrowed computer with
# a GPU. Each 'cell' below runs when you press Shift+Enter. Lines starting with ! are
# shell (terminal) commands, lines starting with % are notebook commands; everything
# else is ordinary Python.

# os = file/folder helpers, getpass = ask for a password without echoing it,
# subprocess = run another program (git) from Python.
import os, getpass, subprocess
# Settings you may change: USE_ZIP = True if you uploaded the repo as a zip file instead
# of giving a GitHub token; ZIP_NAME = the uploaded file's name; REPO = the GitHub repo.
USE_ZIP = False
ZIP_NAME = "VisuallyImpairedBodycam.zip"
REPO = "Ore-Porat/VisuallyImpairedBodycam"

# %cd changes the notebook's working folder (/content is Colab's scratch area).
%cd /content
# Option 1: the repo was uploaded as a zip. Unpack it quietly (-q) and overwrite (-o).
if USE_ZIP:
    !unzip -q -o {ZIP_NAME} -d /content/repo_zip
# os.walk lists every folder; next(...) grabs the first one that has a training/ subfolder.
    # the zip may contain a top-level folder; find the one with training/
    root = next(r for r, d, f in os.walk("/content/repo_zip") if "training" in d)
# Option 2: clone from GitHub, but only if the folder does not already exist (so re-running
# the cell is harmless). The token is typed in a hidden prompt and never stored.
    !rm -rf /content/VisuallyImpairedBodycam && cp -r "{root}" /content/VisuallyImpairedBodycam
# --depth 1 downloads only the latest version, not the whole history (faster).
elif not os.path.isdir("/content/VisuallyImpairedBodycam"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1",
# `del token` forgets the token variable as soon as it is no longer needed.
                    f"https://{token}@github.com/{REPO}.git", "/content/VisuallyImpairedBodycam"], check=True)
    del token
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", "/content/VisuallyImpairedBodycam", "remote", "set-url", "origin",
# Finally move into the repo folder so the following cells can run its scripts.
                    f"https://github.com/{REPO}.git"], check=True)
%cd /content/VisuallyImpairedBodycam

## 2. Install

In [ ]:
# --- Install the training library --------------------------------------------------
# pip is Python's package installer. -q = quiet. ultralytics = the YOLO library,
# albumentations = extra image augmentations (blur etc.), pyyaml = reads .yaml files.
!pip install -q "ultralytics>=8.3" albumentations pyyaml
# Check that a GPU is available (CUDA = NVIDIA's GPU interface). If this prints
# 'CUDA: False', go to Runtime -> Change runtime type and pick a GPU; training on a CPU
# would take many hours.
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 3. Download + convert the dataset (TR-1, TR-2)

Downloads the 5K Groceries annotations (~375 MB) from
github.com/aleksandar-aleksandrov/groceries-object-detection-dataset, converts VOC XML to YOLO and
makes the stratified 80/10/10 split (seed 0) in `data/grocereye/`.

In [ ]:
# --- Download the dataset and convert it to YOLO format -------------------------------
# This runs training/get_grocereye_data.py (open that file for a full explanation). It
# fetches the public grocery photos + box annotations, converts them, and splits them
# into train/val/test folders under data/grocereye/.
!python training/get_grocereye_data.py

## 4. Train (Stage A)

Set `NAME` / `EPOCHS` as needed. For a quick check first, add `--epochs 1 --fraction 0.05`.

In [ ]:
# --- Train the model -----------------------------------------------------------------
# NAME is the run name; the finished weights land in models/<NAME>/best.pt.
# EPOCHS = how many full passes over the training photos. More is slower but usually
# better, up to a point (training stops early by itself if it stops improving).
NAME = "grocereye_a"
EPOCHS = 100
# Run training/train.py (see that file for every option). Curly braces {NAME} insert the
# Python variables above into the shell command. --imgsz 320 = image size fed to the
# network, --batch 64 = images per step, --device 0 = use the first GPU,
# --workers 2 = background threads that load images.
!python training/train.py --name {NAME} --epochs {EPOCHS} --imgsz 320 --batch 64 --device 0 --workers 2

## 5. Evaluate on the test split (TR-4, NFR-5)

In [ ]:
# --- Score the model on the held-out test photos -----------------------------------
# training/evaluate.py measures accuracy (mAP50, precision/recall per class, confusion
# matrix) and writes a markdown report into models/<NAME>/eval_report.md.
!python training/evaluate.py --weights models/{NAME}/best.pt --split test --device 0
# Show that markdown report right here in the notebook, nicely formatted.
from IPython.display import Markdown, display
display(Markdown(open(f"models/{NAME}/eval_report.md").read()))

## 6. Download the weights and report

Put `best.pt` in `models/<NAME>/` on the laptop and commit the eval report (weights are gitignored).
Then set `model.weights: models/<NAME>/best.pt` and `model.classes_file: config/classes.yaml` in
`config/server.yaml`.

In [ ]:
# --- Package the results so you can download them ----------------------------------
# Zip the models/<NAME>/ folder (best.pt + report + plots) into /content/<NAME>.zip.
!cd models && zip -q -r /content/{NAME}.zip {NAME}
# Also copy the dated report(s) from reports/ to /content/. `2>/dev/null || true` hides
# the error and carries on if there is no such file.
!cp reports/eval_{NAME}_*.md /content/ 2>/dev/null || true
# google.colab.files.download pops up the browser's 'save file' dialog for the zip.
from google.colab import files
files.download(f"/content/{NAME}.zip")